In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("AttendanceAnalysis") \
    .getOrCreate()


In [11]:
import pandas as pd

data = [
    ["E001","Sales","2026-05-10 09:00:00","2026-05-10 17:30:00",20],
    ["E002","HR","2026-05-10 09:15:00","2026-05-10 16:45:00",15],
    ["E003","IT","2026-05-10 10:30:00","2026-05-10 14:00:00",6],
    ["E004","Sales","2026-05-10 08:45:00","2026-05-10 18:00:00",25],
    ["E005","HR","2026-05-10 11:30:00","2026-05-10 15:00:00",4],
    ["E006","IT","2026-05-10 09:30:00","2026-05-10 19:00:00",30],
    ["E007","Finance","2026-05-10 10:15:00","2026-05-10 13:30:00",3],
    ["E008","Sales","2026-05-10 09:00:00","2026-05-10 17:00:00",18],
    ["E009","Finance","2026-05-10 12:00:00","2026-05-10 16:00:00",5],
    ["E010","IT","2026-05-10 08:30:00","2026-05-10 17:30:00",22],
    ["E011","HR","2026-05-10 10:45:00","2026-05-10 18:15:00",19],
    ["E012","Sales","2026-05-10 11:15:00","2026-05-10 14:30:00",7],
    ["E013","IT","2026-05-10 09:10:00","2026-05-10 18:40:00",28],
    ["E014","Finance","2026-05-10 09:50:00","2026-05-10 17:20:00",16],
    ["E015","HR","2026-05-10 08:40:00","2026-05-10 16:10:00",14]
]

df = pd.DataFrame(data, columns=[
    "employee_id","department","clock_in","clock_out","tasks_completed"
])

df.to_csv("attendance.csv", index=False)

print("CSV file created successfully!")

CSV file created successfully!


In [12]:
#Load Dataset
df = spark.read.csv(
    "attendance.csv",
    header=True,
    inferSchema=True
)


In [13]:
df = df.withColumn(
    "work_hours",
    (unix_timestamp("clock_out") -
     unix_timestamp("clock_in"))/3600
)


In [14]:
late_logins = df.filter(hour("clock_in") > 10)

absentees = df.filter(col("work_hours") < 4)

late_logins.show()
absentees.show()


+-----------+----------+-------------------+-------------------+---------------+----------+
|employee_id|department|           clock_in|          clock_out|tasks_completed|work_hours|
+-----------+----------+-------------------+-------------------+---------------+----------+
|       E005|        HR|2026-05-10 11:30:00|2026-05-10 15:00:00|              4|       3.5|
|       E009|   Finance|2026-05-10 12:00:00|2026-05-10 16:00:00|              5|       4.0|
|       E012|     Sales|2026-05-10 11:15:00|2026-05-10 14:30:00|              7|      3.25|
+-----------+----------+-------------------+-------------------+---------------+----------+

+-----------+----------+-------------------+-------------------+---------------+----------+
|employee_id|department|           clock_in|          clock_out|tasks_completed|work_hours|
+-----------+----------+-------------------+-------------------+---------------+----------+
|       E003|        IT|2026-05-10 10:30:00|2026-05-10 14:00:00|              6

In [15]:
dept_summary = df.groupBy("department").agg(
    avg("work_hours").alias("avg_hours"),
    avg("tasks_completed").alias("avg_tasks")
)

dept_summary.show()


+----------+-----------------+---------+
|department|        avg_hours|avg_tasks|
+----------+-----------------+---------+
|     Sales|             7.25|     17.5|
|        HR|              6.5|     13.0|
|   Finance|4.916666666666667|      8.0|
|        IT|            7.875|     21.5|
+----------+-----------------+---------+

